# SECCIÓN 1 — Introducción

En esta primera fase se realiza la exploración inicial del dataset M5 Forecasting, con el objetivo de validar su integridad, revisar la coherencia de las series temporales y preparar un conjunto de datos consistente para las siguientes etapas del pipeline.

Se analizan las tres fuentes principales del dataset:

- sales_train_validation.csv: ventas diarias por combinación producto–tienda
- calendar.csv: fechas, eventos y variables temporales
- sell_prices.csv: precios por producto–tienda–semana

Esta exploración permite detectar posibles inconsistencias, valores faltantes, duplicados y validar las relaciones entre tablas. Asimismo, se evalúa la distribución de las ventas y la continuidad temporal del calendario.

Los resultados obtenidos en esta fase constituyen la base para la posterior transformación supervisada, donde se generarán variables derivadas como lags y rolling windows.

Para facilitar la reutilización del código y mantener una estructura modular del proyecto, se definieron funciones auxiliares dentro del directorio src/exploratory, las cuales son utilizadas en los notebooks de exploración.<br>
Se ajusta el directorio de trabajo para permitir la reutilización de funciones definidas en el módulo src, manteniendo una estructura modular del proyecto.

In [1]:
import os
import sys

# Obtener ruta actual
current_path = os.getcwd()

# Subir dos niveles: notebooks → M3
PROJECT_ROOT = os.path.abspath(
    os.path.join(current_path, "..", "..") #porque estamos en la carpeta exploratory/ - de pruebas
)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\Titanio\TFM\M3


# SECCIÓN 2 — Carga de datos

In [3]:
# Importar loader
from src.exploratory.data_loading import load_m5_data
import pandas as pd

# --- CARGA DE DATOS ---
sales, calendar, prices = load_m5_data()

print(sales.shape)

En el script de data_loading se aplicó un proceso de optimización de memoria mediante reducción automática de tipos numéricos (downcasting), permitiendo la manipulación del dataset completo sin errores de memoria.

# SECCIÓN 3 — Vista general

In [4]:
sales.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


In [5]:
calendar.head()

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [6]:
prices.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


# SECCIÓN 4 — Validaciones estructurales

In [7]:
# --- INFO GENERAL ---
print("Sales:", sales.shape)
print("Calendar:", calendar.shape)
print("Prices:", prices.shape)

sales.describe()

Sales: (30490, 1919)
Calendar: (1969, 14)
Prices: (6841121, 4)


,d_1,d_2,d_3,d_4,d_5,d_6,d_7,d_8,d_9,d_10,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
count,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000,...,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000
mean,1.070220,1.041292,0.780026,0.833454,0.627944,0.958052,0.918662,1.244080,1.073663,0.838701,...,1.370581,1.586159,1.693670,1.248245,1.232207,1.159167,1.149000,1.328862,1.605838,1.633158
std,5.126689,5.365468,3.667454,4.415141,3.379344,4.785947,5.059495,6.617729,5.917204,4.206199,...,3.740017,4.097191,4.359809,3.276925,3.125471,2.876026,2.950364,3.358012,4.089422,3.812248
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,2.000000,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000,2.000000
max,360.000000,436.000000,207.000000,323.000000,296.000000,314.000000,316.000000,370.000000,385.000000,353.000000,...,129.000000,160.000000,204.000000,98.000000,100.000000,88.000000,77.000000,141.000000,171.000000,130.000000


El tamaño de las tablas coincide con las especificaciones oficiales del M5 Forecasting.
La tabla de ventas contiene 30.490 series temporales, cada una con 1.913 días de observación.
El calendario cubre 1.969 días consecutivos, lo que confirma la continuidad temporal.
La tabla de precios contiene 6.8 millones de registros, lo cual es coherente con la combinación item–tienda–semana.<br>

describe():<br>
El análisis descriptivo muestra una elevada proporción de valores cero, lo cual es característico de datos de demanda intermitente en entornos retail. 
Las medias bajas reflejan el comportamiento habitual de productos con ventas esporádicas, mientras que los valores máximos corresponden a episodios de alta demanda asociados a determinados productos o eventos.

## Comprobar duplicados

In [8]:
sales["id"].nunique(), sales.shape[0]

(30490, 30490)

## Comprobar tipos de datos

In [10]:
sales.dtypes, calendar.dtypes, prices.dtypes

(id            str
 item_id       str
 dept_id       str
 cat_id        str
 store_id      str
             ...  
 d_1909      int64
 d_1910      int64
 d_1911      int64
 d_1912      int64
 d_1913      int64
 Length: 1919, dtype: object,
 date              str
 wm_yr_wk        int64
 weekday           str
 wday            int64
 month           int64
 year            int64
 d                 str
 event_name_1      str
 event_type_1      str
 event_name_2      str
 event_type_2      str
 snap_CA         int64
 snap_TX         int64
 snap_WI         int64
 dtype: object,
 store_id          str
 item_id           str
 wm_yr_wk        int64
 sell_price    float64
 dtype: object)

# SECCIÓN 5 — Validación temporal<br>
Convertir fecha y revisar rango

In [11]:
calendar["date"] = pd.to_datetime(calendar["date"])
calendar["date"].min(), calendar["date"].max()

(Timestamp('2011-01-29 00:00:00'), Timestamp('2016-06-19 00:00:00'))

## Comprobar continuidad del calendario

In [12]:
calendar["date"].diff().value_counts().head()

date
1 days    1968
Name: count, dtype: int64

# SECCIÓN 6-1 — Valores faltantes (comprobación de nulos)

In [13]:
sales.isna().sum().sort_values(ascending=False).head(10)

id          0
item_id     0
dept_id     0
cat_id      0
store_id    0
state_id    0
d_1         0
d_2         0
d_3         0
d_4         0
dtype: int64

In [14]:
calendar.isna().sum().sort_values(ascending=False).head(10)

event_type_2    1964
event_name_2    1964
event_name_1    1807
event_type_1    1807
date               0
wm_yr_wk           0
year               0
month              0
wday               0
weekday            0
dtype: int64

In [15]:
prices.isna().sum().sort_values(ascending=False).head(10)

store_id      0
item_id       0
wm_yr_wk      0
sell_price    0
dtype: int64

Texto académico para documentar:
La distribución global de ventas presenta una fuerte asimetría hacia cero, con una gran proporción de días sin ventas. Este comportamiento es característico del retail intermitente y confirma la necesidad de modelos robustos a datos dispersos.

Se observan colas largas hacia la derecha, correspondientes a productos con picos de demanda.

# SECCIÓN 6-2 — Exploración de outliers<br>
Detección de valores extremos

In [17]:
import numpy as np

max_values = sales[[c for c in sales.columns if c.startswith("d_")]].max().sort_values(ascending=False).head(10)
max_values


d_960    763
d_959    709
d_938    709
d_337    693
d_511    648
d_98     634
d_329    633
d_908    626
d_957    620
d_330    619
dtype: int64

Los valores máximos observados (hasta ~400 unidades en un día) corresponden a productos con alta rotación o eventos específicos.
No se detectan valores anómalos imposibles (negativos, extremadamente altos), por lo que no se requiere limpieza adicional.

# SECCIÓN 6-3 — Comprobación de valores unicos

In [ ]:
# calendar.d debe ser único
assert calendar["d"].is_unique, "Error: calendar.d debería ser único"

# sales.id debe ser único en formato wide
assert sales["id"].is_unique, "Error: sales.id debería ser único en formato wide"

# prices store-item-semana debe ser único
assert not prices[["store_id","item_id","wm_yr_wk"]].duplicated().any(), \
       "Error: la combinación store-item-semana debería ser única"

Los precios se reportan una vez por semana
Para cada combinación store–item–semana debe haber un único precio
Si hubiera duplicados → merge incorrecto → leakage → modelo roto


La tabla de ventas no contiene valores nulos.
En el calendario, los nulos se concentran en columnas de eventos (event_name_1, event_type_1, etc.), lo cual es esperado, ya que la mayoría de días no tienen eventos asociados.
La tabla de precios no presenta valores faltantes.

No se requiere imputación en esta fase.

# SECCIÓN 7 QUINQUIES — Integridad entre tablas<br>
Comprobar que todos los días de sales están en calendar

In [18]:
days_sales = [c for c in sales.columns if c.startswith("d_")]
days_calendar = calendar["d"].tolist()

set(days_sales) - set(days_calendar)

set()

Se verifica que todos los identificadores de día (d_1 a d_1913) presentes en la tabla de ventas están correctamente definidos en la tabla de calendario.
Esto garantiza que el merge posterior será consistente.

# SECCIÓN 8 — Exploración de distribuciones (ventas)

In [ ]:
import matplotlib.pyplot as plt

sales_values = sales[[c for c in sales.columns if c.startswith("d_")]].values.flatten()

plt.figure(figsize=(10,5))
plt.hist(sales_values, bins=50, color="steelblue")
plt.title("Distribución global de ventas (todas las series, todos los días)")
plt.xlabel("Unidades vendidas")
plt.ylabel("Frecuencia")
plt.show()

# SECCIÓN 9 — Hallazgos clave 

Hallazgos principales de la exploración inicial:<br>
El dataset presenta una estructura coherente y sin inconsistencias críticas.<br>
La distribución de ventas es altamente esparsa, con predominio de ceros.<br>
No se detectan valores faltantes relevantes ni outliers anómalos.<br>
El calendario es continuo y cubre todo el rango temporal de ventas.<br>
Las claves principales (id, d, store-item-semana) son únicas y válidas.<br>
El dataset está listo para la fase de Feature Engineering, donde se generarán lags, rolling windows y codificación categórica.

# SECCIÓN 10 — Guardar dataset limpio

In [19]:
print("Resumen final antes de guardar:")
print("- Ventas:", sales.shape)
print("- Calendario:", calendar.shape)
print("- Precios:", prices.shape)
print("Sin nulos en ventas:", sales.isna().sum().sum() == 0)
print("Sin nulos en precios:", prices.isna().sum().sum() == 0)
print("Nulos esperados en calendario:", calendar.isna().sum().sum())


Resumen final antes de guardar:
- Ventas: (30490, 1919)
- Calendario: (1969, 14)
- Precios: (6841121, 4)
Sin nulos en ventas: True
Sin nulos en precios: True
Nulos esperados en calendario: 7542


Tras completar la exploración inicial y las validaciones estructurales, se confirma que las tres tablas del M5 presentan una estructura coherente y sin inconsistencias críticas.

La distribución de ventas es altamente esparsa, característica del retail intermitente, y no se detectan valores anómalos ni outliers imposibles.
El calendario es continuo y cubre todo el rango temporal de ventas, mientras que la tabla de precios presenta una estructura semanal consistente.

Con estas comprobaciones, los datasets están listos para ser almacenados en formato Parquet y utilizados en la fase de Feature Engineering.

In [20]:
import os
os.getcwd()


'C:\\Users\\Titanio\\TFM\\M3'

El almacenamiento en formato Parquet permite una lectura eficiente y optimizada en las fases posteriores del pipeline, especialmente cuando se utilizan motores distribuidos como Apache Spark.

In [22]:
os.makedirs("data/processed/exploratory", exist_ok=True)
sales.to_parquet("data/processed/exploratory/sales_clean.parquet", index=False)
calendar.to_parquet("data/processed/exploratory/calendar_clean.parquet", index=False)
prices.to_parquet("data/processed/exploratory/prices_clean.parquet", index=False)

print("Datasets limpios guardados.")

Datasets limpios guardados.


# SECCIÓN 12 — Limitaciones y transición a procesamiento distribuido
Durante la exploración inicial y las primeras pruebas de transformación se identificó que el volumen del dataset y la estructura de las series temporales generan un consumo elevado de memoria cuando se utilizan herramientas basadas únicamente en pandas.<br>
En particular, operaciones como la reestructuración del dataset (transformación de formato wide a long), la integración con tablas auxiliares y la generación de variables temporales requieren el manejo simultáneo de decenas de millones de registros.<br>
Estas limitaciones motivaron la adopción de Apache Spark como tecnología principal para la fase de transformación masiva de datos. Spark permite ejecutar las mismas transformaciones de forma distribuida y paralela, manteniendo la lógica validada previamente en este entorno exploratorio.<br>
En consecuencia, las operaciones definidas en este notebook se implementarán posteriormente dentro de un pipeline escalable basado en Spark.